# Stich Great Lake DEM

In [ ]:
import os, glob
import rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling

Modeltop = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM"
target_crs = "EPSG:3174"
target_res = 30
nodata_out = -9999.0

tmp_dir = os.path.join(Modeltop, "_tmp_reproj_3174")
os.makedirs(tmp_dir, exist_ok=True)

tifs = sorted(glob.glob(os.path.join(Modeltop, "*.tif")))
if not tifs:
    raise FileNotFoundError(f"No .tif files found in: {Modeltop}")

reproj_paths = []

for fp in tifs:
    with rasterio.open(fp) as src:
        if src.crs is None:
            raise ValueError(f"{os.path.basename(fp)} has no CRS. Define Projection first.")

        src_nodata = src.nodata
        out_fp = os.path.join(tmp_dir, os.path.splitext(os.path.basename(fp))[0] + "_3174.tif")

        if not os.path.exists(out_fp):
            transform, width, height = calculate_default_transform(
                src.crs, target_crs, src.width, src.height, *src.bounds
            )

            meta = src.meta.copy()
            meta.update({
                "crs": target_crs,
                "transform": transform,
                "width": width,
                "height": height,
                "dtype": "float32",
                "nodata": nodata_out,
                "compress": "lzw",
                "tiled": True,
                "BIGTIFF": "IF_SAFER"
            })

            with rasterio.open(out_fp, "w", **meta) as dst:
                reproject(
                    source=rasterio.band(src, 1),
                    destination=rasterio.band(dst, 1),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src_nodata,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    dst_nodata=nodata_out,
                    resampling=Resampling.bilinear
                )

        reproj_paths.append(out_fp)

# Merge and write directly to disk (BigTIFF)
out_tif = os.path.join(Modeltop, "DEM_merged_3174_30m.tif")

srcs = [rasterio.open(fp) for fp in reproj_paths]
template = srcs[0].meta.copy()
template.update({
    "driver": "GTiff",
    "crs": target_crs,
    "dtype": "float32",
    "nodata": nodata_out,
    "compress": "lzw",
    "tiled": True,
    "BIGTIFF": "YES"
})

merge(
    srcs,
    res=(target_res, target_res),
    nodata=nodata_out,
    resampling=Resampling.bilinear,
    target_aligned_pixels=True,
    dst_path=out_tif,
    dst_kwds=template
)

for s in srcs:
    s.close()

print("✅ Wrote:", out_tif)


In [1]:
import arcpy
arcpy.env.overwriteOutput = True

out_tif = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"

arcpy.management.CalculateStatistics(out_tif)
arcpy.management.BuildPyramids(out_tif)
print("✅ Built statistics + pyramids")

✅ Built statistics + pyramids


# Stich the Lakes

In [ ]:
out_dir  = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers"


In [ ]:
import os
import arcpy

Lakes_path = r"D:\Users\abolmaal\Arcgis\GreatLakes"
out_shp = os.path.join(Lakes_path, "hydro_p_GreatLakes.shp")

arcpy.env.workspace = Lakes_path
arcpy.env.overwriteOutput = True

# Get all shapefiles in the folder
shps = arcpy.ListFeatureClasses(feature_type="All")
shps = [os.path.join(Lakes_path, s) for s in shps if s.lower().endswith(".shp")]

if not shps:
    raise FileNotFoundError(f"No shapefiles found in: {Lakes_path}")

# Merge
arcpy.management.Merge(shps, out_shp)

print(f"✅ Merged {len(shps)} shapefiles -> {out_shp}")


✅ Merged 5 shapefiles -> D:\Users\abolmaal\Arcgis\GreatLakes\hydro_p_GreatLakes.shp


# Create constant heads boundary 


to make a constant-head (CHD) boundary from your basin DEM, you basically do two things:

1- decide which grid cells are “constant head” (usually the Great Lakes water cells, and sometimes the outer model boundary), and

2- assign a head value (stage elevation) to those cells for each stress period.

### 1) Rasterize your hydro polygons to your 30 m model grid (aligned to your DEM)

This makes a water_30m.tif where water cell = -1, outside = 0, inside (land) = 1

In [6]:
import os
import arcpy
from arcpy import sa

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# ----------------------------
# INPUTS
# ----------------------------
hydro_p = r"D:\Users\abolmaal\Arcgis\GreatLakes\hydro_p_GreatLakes.shp"
dem = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"
boundary_p = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"

out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads"
os.makedirs(out_dir, exist_ok=True)

buffer_dist_m = 20000  # 20 km buffer

# ----------------------------
# ENV ALIGNMENT
# ----------------------------
arcpy.env.snapRaster = dem
arcpy.env.cellSize   = dem
arcpy.env.extent     = dem
arcpy.env.outputCoordinateSystem = arcpy.Describe(dem).spatialReference

dem_r  = arcpy.Raster(dem)
dem_sr = arcpy.Describe(dem).spatialReference

print("DEM rows/cols:", dem_r.height, dem_r.width)
print("DEM cellsize:", dem_r.meanCellWidth, dem_r.meanCellHeight)
print("DEM SR:", dem_sr.name, "factoryCode:", dem_sr.factoryCode)

# ----------------------------
# HELPERS
# ----------------------------
def safe_delete(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)

def project_if_needed(fc, out_name):
    sr = arcpy.Describe(fc).spatialReference
    if sr.factoryCode != dem_sr.factoryCode:
        out_fc = os.path.join(out_dir, out_name)
        safe_delete(out_fc)
        arcpy.management.Project(fc, out_fc, dem_sr)
        return out_fc
    return fc

def ensure_constant_field(fc, field_name, value=1):
    fset = {f.name.upper() for f in arcpy.ListFields(fc)}
    if field_name.upper() not in fset:
        arcpy.management.AddField(fc, field_name, "SHORT")
    arcpy.management.CalculateField(fc, field_name, str(value), "PYTHON3")
    return field_name

# ----------------------------
# PROJECT INPUTS
# ----------------------------
hydro_use = project_if_needed(hydro_p, "hydro_proj.shp")
bnd_use   = project_if_needed(boundary_p, "boundary_proj.shp")

# ----------------------------
# FIX GEOMETRY (optional but helps with weird gaps)
# ----------------------------
try:
    arcpy.management.RepairGeometry(bnd_use)
except Exception as e:
    print("⚠️ RepairGeometry skipped/failed:", e)

# ----------------------------
# 2 KM BUFFER AROUND BOUNDARY
# ----------------------------
bnd_buff = os.path.join(out_dir, f"boundary_buffer_{buffer_dist_m}m.shp")
safe_delete(bnd_buff)

# dissolve_option="ALL" prevents internal seams
arcpy.analysis.Buffer(
    in_features=bnd_use,
    out_feature_class=bnd_buff,
    buffer_distance_or_field=f"{buffer_dist_m} Meters",
    dissolve_option="ALL"
)

print("✅ Buffered boundary saved:", bnd_buff)

# ----------------------------
# RASTERIZE BUFFERED BOUNDARY: inside=1, outside=0
# ----------------------------
bnd_field = ensure_constant_field(bnd_buff, "BNDVAL", 1)

bnd_raw = os.path.join(out_dir, "boundary_buff_raw.tif")
safe_delete(bnd_raw)

arcpy.conversion.PolygonToRaster(
    in_features=bnd_buff,
    value_field=bnd_field,
    out_rasterdataset=bnd_raw,
    cell_assignment="MAXIMUM_AREA",
    priority_field="NONE",
    cellsize=dem
)

bnd_ras = sa.Int(sa.Con(sa.IsNull(bnd_raw), 0, 1))  # 0/1 everywhere

# ----------------------------
# RASTERIZE WATER: water=1, else 0
# ----------------------------
wat_field = ensure_constant_field(hydro_use, "WATVAL", 1)

wat_raw = os.path.join(out_dir, "water_raw.tif")
safe_delete(wat_raw)

arcpy.conversion.PolygonToRaster(
    in_features=hydro_use,
    value_field=wat_field,
    out_rasterdataset=wat_raw,
    cell_assignment="MAXIMUM_AREA",
    priority_field="NONE",
    cellsize=dem
)

water_ras = sa.Int(sa.Con(sa.IsNull(wat_raw), 0, 1))

# ----------------------------
# FINAL MASK:
# outside buffered boundary = 0
# inside buffered boundary (non-water) = 1
# water = -1  (only within buffered boundary)
# ----------------------------
base  = bnd_ras
final = sa.Con(water_ras == 1, -1, base)
final = sa.Con(bnd_ras == 0, 0, final)   # enforce outside=0
final = sa.Int(final)

# ----------------------------
# SAVE TO FILE GEODATABASE
# ----------------------------
gdb = os.path.join(out_dir, "rasters.gdb")
if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(out_dir, "rasters.gdb")

out_gdb_ras = os.path.join(gdb, f"domain_water_mask_30m_buff{buffer_dist_m}m")
safe_delete(out_gdb_ras)

arcpy.management.CopyRaster(
    in_raster=final,
    out_rasterdataset=out_gdb_ras,
    pixel_type="16_BIT_SIGNED",
    nodata_value="0"
)

print("✅ Saved to GDB raster:", out_gdb_ras)

# ----------------------------
# OPTIONAL: export to TIFF (may fail if huge)
# ----------------------------
out_tif = os.path.join(out_dir, f"domain_water_mask_30m_buff{buffer_dist_m}m.tif")
safe_delete(out_tif)

try:
    arcpy.management.CopyRaster(
        in_raster=out_gdb_ras,
        out_rasterdataset=out_tif,
        pixel_type="16_BIT_SIGNED",
        nodata_value="0",
        format="TIFF"
    )
    print("✅ Exported TIFF:", out_tif)
except Exception as e:
    print("⚠️ TIFF export failed (likely too large). Keep the GDB raster.")
    print("   Error:", e)

# ----------------------------
# SMALL SAMPLE CHECK
# ----------------------------
import numpy as np
sample = arcpy.RasterToNumPyArray(out_gdb_ras, ncols=500, nrows=500, nodata_to_value=999)
vals = np.unique(sample[sample != 999])
print("✅ sample unique values:", vals)


DEM rows/cols: 49758 66126
DEM cellsize: 30.0 30.0
DEM SR: NAD_1983_Great_Lakes_Basin_Albers factoryCode: 3174
✅ Buffered boundary saved: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\boundary_buffer_20000m.shp
✅ Saved to GDB raster: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\rasters.gdb\domain_water_mask_30m_buff20000m
✅ Exported TIFF: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff20000m.tif
✅ sample unique values: []


# Making HK
Below is a single ArcPy script that does everything end-to-end:

1- Clip surface geology to extended_Buffer2km

2- Add UPKh_m_d, MidKh_m_d, LowKh_m_d

3-Set those = 100 where intersecting hydro_p

4- Convert Excel → table (inside a File GDB)

5- Create robust text join keys (avoids numeric/text mismatch)

6- Join Excel fields onto the final output

7- Export a final feature class (and optional shapefile)

Important note: Joining Excel can add many fields and shapefiles have field name limits (10 chars). I output to a File Geodatabase feature class by default (recommended). You can still export a shapefile if you want.


# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------
surfacegeology = r"S:\Data\Other_Data\Xu_2021\Data\09_Integrated_Surficial_Geology_Map\Integrated_surficial_geology_mapping.shp"
extended_Buffer2km = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\boundary_buffer_2000m.shp"
hydro_p = r"D:\Users\abolmaal\Arcgis\GreatLakes\hydro_p_GreatLakes.shp"

excel_xlsx = r"S:\Data\Other_Data\Xu_2021\Data\11_Quaternary_Material_Description_Calibrated_Kh\Calibrated_surficial_Kh.xlsx"
EXCEL_SHEET_NAME = "GLB_surf_dissolve_merged"


In [20]:
import os
import arcpy

arcpy.env.overwriteOutput = True

# ============================================================
# INPUTS
# ============================================================
surfacegeology = r"S:\Data\Other_Data\Xu_2021\Data\09_Integrated_Surficial_Geology_Map\Integrated_surficial_geology_mapping.shp"

# Use THIS CSV (your file)
csv_path = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\Calibrated_surficial_Kh__ALLCOLS.csv"

extended_Buffer2km = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\boundary_buffer_2000m.shp"
hydro_p = r"D:\Users\abolmaal\Arcgis\GreatLakes\hydro_p_GreatLakes.shp"

# If CSV does NOT contain Lower_ms, create it using this constant:
LOWER_DEFAULT = 0.086

# Buffer the lake polygon slightly so "near-miss" polygons are included
HYDRO_BUFFER_DIST = "25 Meters"   # try "10 Meters", "25 Meters", "50 Meters"

# ============================================================
# OUTPUTS
# ============================================================
out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK"
os.makedirs(out_dir, exist_ok=True)

gdb = os.path.join(out_dir, "HK_join_clip.gdb")
if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(out_dir, os.path.basename(gdb))

surf_fc   = os.path.join(gdb, "surfacegeology_base")
csv_tbl   = os.path.join(gdb, "Calibrated_surficial_Kh_tbl")
joined_fc = os.path.join(gdb, "surfacegeo_joined")
clip_fc   = os.path.join(gdb, "surfacegeo_joined_clip2km")

final_shp = os.path.join(out_dir, "surfacegeology_Kh_clip2km_Rech.shp")

# ============================================================
# HELPERS
# ============================================================
def same_sr(sr1, sr2):
    try:
        if sr1.factoryCode and sr2.factoryCode:
            return sr1.factoryCode == sr2.factoryCode
    except Exception:
        pass
    return (sr1.name or "") == (sr2.name or "")

def ensure_field(dataset, name, ftype, length=None):
    existing = {f.name for f in arcpy.ListFields(dataset)}
    if name not in existing:
        if ftype.upper() == "TEXT" and length is not None:
            arcpy.management.AddField(dataset, name, ftype, field_length=length)
        else:
            arcpy.management.AddField(dataset, name, ftype)

def to_key_py(v):
    if v is None:
        return None
    s = str(v).strip()
    if s == "":
        return None
    # convert 450.0 -> 450
    try:
        fv = float(s)
        if fv.is_integer():
            s = str(int(fv))
        else:
            s = str(fv)
    except Exception:
        pass
    return s.strip().upper()

def make_join_id(dataset, source_field, join_field="JOIN_ID"):
    ensure_field(dataset, join_field, "TEXT", length=50)
    codeblock = r"""
def to_key(v):
    if v is None:
        return None
    s = str(v).strip()
    if s == "":
        return None
    try:
        fv = float(s)
        if fv.is_integer():
            s = str(int(fv))
        else:
            s = str(fv)
    except:
        pass
    return s.strip().upper()
"""
    arcpy.management.CalculateField(
        dataset, join_field,
        expression=f"to_key(!{source_field}!)",
        expression_type="PYTHON3",
        code_block=codeblock
    )

def count_intersections(fc, fc_field, tbl, tbl_field):
    fc_keys = set()
    with arcpy.da.SearchCursor(fc, [fc_field]) as cur:
        for (v,) in cur:
            k = to_key_py(v)
            if k is not None:
                fc_keys.add(k)

    tb_keys = set()
    with arcpy.da.SearchCursor(tbl, [tbl_field]) as cur:
        for (v,) in cur:
            k = to_key_py(v)
            if k is not None:
                tb_keys.add(k)

    return len(fc_keys.intersection(tb_keys)), len(fc_keys), len(tb_keys)

# ============================================================
# 1) Copy surfacegeology to GDB
# ============================================================
print("1/8 Copy surfacegeology to GDB...")
if arcpy.Exists(surf_fc):
    arcpy.management.Delete(surf_fc)
arcpy.management.CopyFeatures(surfacegeology, surf_fc)

surf_fields = [f.name for f in arcpy.ListFields(surf_fc)]
if "POLYID" not in surf_fields:
    raise RuntimeError(f"POLYID not found in surfacegeology. Fields: {surf_fields}")
print("  ✅", surf_fc)

# ============================================================
# 2) Import CSV to GDB table
# ============================================================
print("2/8 Import CSV -> GDB table...")
if arcpy.Exists(csv_tbl):
    arcpy.management.Delete(csv_tbl)
arcpy.conversion.TableToTable(csv_path, gdb, os.path.basename(csv_tbl))
print("  ✅", csv_tbl)

tbl_fields = [f.name for f in arcpy.ListFields(csv_tbl)]
print("  CSV fields found:", tbl_fields)

# required columns
if "Upper_ms" not in tbl_fields or "Middle_ms" not in tbl_fields:
    raise RuntimeError(
        f"CSV table must contain Upper_ms and Middle_ms.\nFields found: {tbl_fields}"
    )

# Lower_ms may be missing; create it if needed
if "Lower_ms" not in tbl_fields:
    print(f"  ⚠️ Lower_ms not found. Creating Lower_ms = {LOWER_DEFAULT} ...")
    ensure_field(csv_tbl, "Lower_ms", "DOUBLE")
    arcpy.management.CalculateField(csv_tbl, "Lower_ms", str(LOWER_DEFAULT), "PYTHON3")
    tbl_fields = [f.name for f in arcpy.ListFields(csv_tbl)]

# ensure join candidates exist
candidates = [c for c in ["col_1", "col_2"] if c in tbl_fields]
if not candidates:
    raise RuntimeError(f"CSV table missing col_1/col_2. Fields found: {tbl_fields}")

# ============================================================
# 3) Determine which CSV column matches POLYID (col_1 vs col_2)
# ============================================================
print("3/8 Determine join key (POLYID <-> col_1 or col_2)...")
best_field, best_match = None, -1
debug = []
for c in candidates:
    m, ns, nt = count_intersections(surf_fc, "POLYID", csv_tbl, c)
    debug.append((c, m, ns, nt))
    if m > best_match:
        best_match = m
        best_field = c

print("  Overlap tests (candidate, matches, surf_unique, table_unique):")
for row in debug:
    print("   ", row)

if best_match <= 0:
    raise RuntimeError(
        "No matches found between surface POLYID and CSV col_1/col_2.\n"
        "This means the IDs truly differ (not just formatting)."
    )

print(f"  ✅ Using CSV join field: {best_field} (matches={best_match})")

# ============================================================
# 4) Join ALL CSV columns onto surfacegeology
# ============================================================
print("4/8 Join CSV -> surfacegeology (keep all CSV columns)...")
make_join_id(surf_fc, "POLYID", "JOIN_ID")
make_join_id(csv_tbl, best_field, "JOIN_ID")

if arcpy.Exists(joined_fc):
    arcpy.management.Delete(joined_fc)
arcpy.management.CopyFeatures(surf_fc, joined_fc)

# Join all fields except OID/Geometry/JOIN_ID
csv_fields_to_add = []
for f in arcpy.ListFields(csv_tbl):
    if f.type in ("OID", "Geometry"):
        continue
    if f.name.lower() == "join_id":
        continue
    csv_fields_to_add.append(f.name)

arcpy.management.JoinField(joined_fc, "JOIN_ID", csv_tbl, "JOIN_ID", csv_fields_to_add)
print(f"  ✅ Joined {len(csv_fields_to_add)} CSV columns.")

# ============================================================
# 5) Create your three Kh columns and set equal to Upper/Middle/Lower
# ============================================================
print("5/8 Create UPKh_m_d, MidKh_m_d, LowKh_m_d...")

for fld in ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d"]:
    ensure_field(joined_fc, fld, "DOUBLE")

# robust conversion (handles text/numeric)
codeblock = r"""
def to_float(v):
    if v is None:
        return None
    try:
        s = str(v).strip()
        if s == "":
            return None
        return float(s)
    except:
        return None
"""
arcpy.management.CalculateField(joined_fc, "UPKh_m_d",  "to_float(!Upper_ms!)",  "PYTHON3", codeblock)
arcpy.management.CalculateField(joined_fc, "MidKh_m_d", "to_float(!Middle_ms!)", "PYTHON3", codeblock)
arcpy.management.CalculateField(joined_fc, "LowKh_m_d", "to_float(!Lower_ms!)",  "PYTHON3", codeblock)

# ============================================================
# 6) Clip to the 2 km buffer polygon
# ============================================================
print("6/8 Clip to 2 km buffer polygon...")

fc_sr  = arcpy.Describe(joined_fc).spatialReference
buf_sr = arcpy.Describe(extended_Buffer2km).spatialReference

buf_for_clip = extended_Buffer2km
tmp_buf = None
if not same_sr(fc_sr, buf_sr):
    tmp_buf = os.path.join("in_memory", "buf_proj")
    arcpy.management.Project(extended_Buffer2km, tmp_buf, fc_sr)
    buf_for_clip = tmp_buf

if arcpy.Exists(clip_fc):
    arcpy.management.Delete(clip_fc)
arcpy.analysis.Clip(joined_fc, buf_for_clip, clip_fc)

if tmp_buf:
    arcpy.management.Delete(tmp_buf)

print("  ✅", clip_fc)

# ============================================================
# 7) Add Rech_m_d and set = 100 for overlaps with hydro_p (buffered)
# ============================================================
print("7/8 Add Rech_m_d and set 100 for lake overlaps (buffered)...")

ensure_field(clip_fc, "Rech_m_d", "DOUBLE")
arcpy.management.CalculateField(clip_fc, "Rech_m_d", "0", "PYTHON3")

# Repair geometry to reduce missed selections
arcpy.management.RepairGeometry(clip_fc)

hyd_sr = arcpy.Describe(hydro_p).spatialReference
hyd_for_sel = hydro_p
tmp_hyd = None
if not same_sr(fc_sr, hyd_sr):
    tmp_hyd = os.path.join("in_memory", "hyd_proj")
    arcpy.management.Project(hydro_p, tmp_hyd, fc_sr)
    hyd_for_sel = tmp_hyd

# Repair + buffer hydro polygon slightly to catch slivers/gaps
arcpy.management.RepairGeometry(hyd_for_sel)

hyd_buf = os.path.join("in_memory", "hyd_buf")
arcpy.analysis.Buffer(
    in_features=hyd_for_sel,
    out_feature_class=hyd_buf,
    buffer_distance_or_field=HYDRO_BUFFER_DIST,
    dissolve_option="ALL"
)

geo_lyr = "geo_lyr"
arcpy.management.MakeFeatureLayer(clip_fc, geo_lyr)

# Select polygons intersecting buffered hydro polygon
arcpy.management.SelectLayerByLocation(
    in_layer=geo_lyr,
    overlap_type="INTERSECT",
    select_features=hyd_buf,
    selection_type="NEW_SELECTION"
)

sel_n = int(arcpy.management.GetCount(geo_lyr)[0])
print(f"  ✅ Selected for Rech_m_d=100: {sel_n}")

arcpy.management.CalculateField(geo_lyr, "Rech_m_d", "100", "PYTHON3")
arcpy.management.SelectLayerByAttribute(geo_lyr, "CLEAR_SELECTION")

# cleanup
arcpy.management.Delete(hyd_buf)
if tmp_hyd:
    arcpy.management.Delete(tmp_hyd)

# ============================================================
# 8) Export final shapefile
# ============================================================
print("8/8 Export final shapefile...")

# drop helper join id if present
if "JOIN_ID" in [f.name for f in arcpy.ListFields(clip_fc)]:
    arcpy.management.DeleteField(clip_fc, ["JOIN_ID"])

if arcpy.Exists(final_shp):
    arcpy.management.Delete(final_shp)
arcpy.conversion.FeatureClassToFeatureClass(clip_fc, out_dir, os.path.basename(final_shp))

print("\nDONE ✅")
print("Final shapefile:", final_shp)


1/8 Copy surfacegeology to GDB...
  ✅ D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HK_join_clip.gdb\surfacegeology_base
2/8 Import CSV -> GDB table...
  ✅ D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HK_join_clip.gdb\Calibrated_surficial_Kh_tbl
  CSV fields found: ['OBJECTID', 'col_1', 'col_2', 'col_3', 'Upper_ms', 'Middle_ms']
  ⚠️ Lower_ms not found. Creating Lower_ms = 0.086 ...
3/8 Determine join key (POLYID <-> col_1 or col_2)...
  Overlap tests (candidate, matches, surf_unique, table_unique):
    ('col_1', 52, 52, 53)
    ('col_2', 2, 52, 52)
  ✅ Using CSV join field: col_1 (matches=52)
4/8 Join CSV -> surfacegeology (keep all CSV columns)...
  ✅ Joined 6 CSV columns.
5/8 Create UPKh_m_d, MidKh_m_d, LowKh_m_d...
6/8 Clip to 2 km buffer polygon...
  ✅ D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HK_join_clip.gdb\surfacegeo_joined_clip2km
7/8 Add Rech_m_d and set 100 for lake overlaps (buffered)...
  ✅ Selected fo